# Homework 4: DevOps and Observability for AI-Built Apps
Order Tracker incident workflow: run the app, instrument it, build the telemetry pipeline, alert on 5xx, and let a headless agent respond.

Run this notebook from the root of your **order-tracker fork**. Each section runs the checks for one question and prints the evidence you need for the answer. Edit the config cell if your ports or metric names differ.

In [ ]:
import json, subprocess, time, urllib.request, urllib.error, base64, pathlib

APP = "http://localhost:8000"
RESPONDER = "http://localhost:8001"
PROM = "http://localhost:9090"
GRAFANA = "http://localhost:3000"
GRAFANA_AUTH = ("admin", "admin")          # change if you set other credentials
INCIDENT_DIR = pathlib.Path("incident-response/incidents")  # where your responder saves evidence

def sh(cmd, timeout=300):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def http(url, method="GET", data=None, headers=None, auth=None):
    h = dict(headers or {})
    if auth:
        h["Authorization"] = "Basic " + base64.b64encode(f"{auth[0]}:{auth[1]}".encode()).decode()
    body = json.dumps(data).encode() if data is not None else None
    if body: h.setdefault("Content-Type", "application/json")
    req = urllib.request.Request(url, data=body, method=method, headers=h)
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.status, resp.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()
    except Exception as e:
        return None, str(e)

print(sh("docker --version"), "|", sh("docker compose version"))

## Question 1: Run the app

In [ ]:
print(sh("docker compose up --build -d --wait", timeout=900))
status, body = http(f"{APP}/healthz")
print(status, body)

**Answer 1:** whatever `/healthz` returned above (expected `{"status":"ok"}`).

## Question 2: Instrument one endpoint
Ask your coding agent to add OpenTelemetry metrics, logs and traces (console exporters) with route and status code on the request metric. Then rebuild and look up `standard-1001`.

In [ ]:
print(sh("docker compose up --build -d --wait", timeout=900))
status, body = http(f"{APP}/api/orders/standard-1001")
print("HTTP status from curl:", status)
print(body)
time.sleep(12)  # console metric exporters flush on an interval
logs = sh("docker compose logs app --no-log-prefix --tail 400")
lines = [l for l in logs.splitlines() if "status" in l.lower() and ("http" in l.lower() or "request" in l.lower())]
print("\n".join(lines[-15:]) or "No matching metric lines yet; print `logs` and search manually.")

**Answer 2:** the status code attribute on the request metric for this lookup.

## Question 3: Telemetry pipeline
Collector + Prometheus + Loki + Tempo + Grafana in Compose. Rebuild, call `standard-1002`, then check all three signals.

In [ ]:
print(sh("docker compose up --build -d --wait", timeout=900))
status, body = http(f"{APP}/api/orders/standard-1002")
print("HTTP status from curl:", status)
time.sleep(20)  # allow scrape + export

METRIC = "http_server_requests_total"   # adjust to the metric name your app/collector exposes
s, r = http(f"{PROM}/api/v1/query?query=" + urllib.request.quote(f"sum by (http_route, http_status_code) ({METRIC})"))
print("Prometheus:", s); print(r[:2000])

s, r = http("http://localhost:3100/loki/api/v1/query_range?query=" + urllib.request.quote('{service_name=~".+"}') + "&limit=5")
print("Loki:", s); print(r[:800])

s, r = http("http://localhost:3200/api/search?limit=5")
print("Tempo:", s); print(r[:800])

**Answer 3:** the `http_status_code` label on the `standard-1002` series. Confirm in Grafana Explore that the log and trace also appear.

## Question 4: 5xx alert
Check the alert state after calling `standard-1002` again.

In [ ]:
http(f"{APP}/api/orders/standard-1002")
time.sleep(60)  # wait for at least one evaluation interval

s, r = http(f"{GRAFANA}/api/prometheus/grafana/api/v1/rules", auth=GRAFANA_AUTH)
print("Grafana rules API:", s)
try:
    data = json.loads(r)
    for g in data["data"]["groups"]:
        for rule in g["rules"]:
            print(f'{rule["name"]:40} state={rule["state"]:10} health={rule.get("health")}')
except Exception:
    print(r[:1500])

**Answer 4:** the state printed for the 5xx rule (Normal / Firing / Pending / No data). Grafana's API labels Normal as `inactive`.

## Question 5: Automatic responder
Start the responder (port 8001), then send the test alert and read what the agent said.

In [ ]:
# Start your responder in a separate terminal first, e.g.:  cd incident-response && uv run python main.py
payload = {"alerts":[{"status":"firing",
  "labels":{"alertname":"ResponderTest","test":"true"},
  "annotations":{"summary":"Test notification; no incident to fix"}}]}
print(http(f"{RESPONDER}/alerts", method="POST", data=payload))

In [ ]:
# Wait for the agent to finish, then read its newest saved output.
time.sleep(60)
files = sorted(INCIDENT_DIR.rglob("*"), key=lambda p: p.stat().st_mtime, reverse=True) if INCIDENT_DIR.exists() else []
files = [f for f in files if f.is_file()]
for f in files[:8]: print(f)
agent_out = next((f for f in files if any(k in f.name.lower() for k in ("response","answer","agent","output","result"))), None)
if agent_out:
    text = agent_out.read_text().strip()
    print("\n--- agent output ---\n" + text)
    print("\nLAST LINE:", text.splitlines()[-1])
else:
    print("Agent output file not found. Check the responder console or adjust INCIDENT_DIR.")

**Answer 5:** paste the agent's response and its last line exactly as printed.

## Question 6: Full flow with the real alert
Trigger `express-1002`, watch the alert fire and the responder start, then verify the fix.

In [ ]:
for i in range(6):
    s, b = http(f"{APP}/api/orders/express-1002")
    print(i, s, b[:150])
    time.sleep(10)

In [ ]:
# Poll the alert until it fires, then check the responder saved a new incident.
for _ in range(12):
    s, r = http(f"{GRAFANA}/api/prometheus/grafana/api/v1/rules", auth=GRAFANA_AUTH)
    try:
        states = [(x["name"], x["state"]) for g in json.loads(r)["data"]["groups"] for x in g["rules"]]
    except Exception:
        states = r[:200]
    print(states)
    if any(st == "firing" for _, st in (states if isinstance(states, list) else [])): break
    http(f"{APP}/api/orders/express-1002")
    time.sleep(15)

In [ ]:
# After the agent has fixed the code: rebuild and verify.
print(sh("docker compose up --build -d --wait", timeout=900))
s, b = http(f"{APP}/api/orders/express-1002")
print("After fix:", s, b[:300])
print(sh("git status --short && git log --oneline -5"))

**Answer 6:** use the agent's root-cause summary and the traceback in `docker compose logs app` for `express-1002`. Match it to one of the four options; don't guess.

## Before you submit
- Commit and push: Collector/Prometheus/Loki/Tempo/Grafana config, dashboard and alert provisioning, `incident-response/`, incident evidence, and the agent's fix.
- Submit the repo link on the course platform.